# 🌊 HWAT 4.7M — Harmonic Wavelet Attention Transformer
## Entraînement sur Kaggle GPU (P100 gratuit)

Modèle : 4.7M params, dim=256, 4 couches, 4 têtes
Données : 100K exemples structurés (synonymes + relations)
Durée estimée : ~5-10 minutes sur GPU

In [ ]:
import math, time, random, os
from pathlib import Path
from collections import Counter
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('='*60)
print('  🌊 HWAT Training — Kaggle GPU')
print(f'  Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('='*60)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONSTANTES & UTILITAIRES
# ═══════════════════════════════════════════════════════════

PHI = 1.618033988749895
ALPHA = 1.0 / PHI
TAU = 2.0 * math.pi

def _fnv1a(s: str) -> int:
    h = 2166136261
    for ch in s.encode('utf-8'):
        h ^= ch; h = (h * 16777619) & 0xFFFFFFFF
    return h

# ═══════════════════════════════════════════════════════════
# PHASE ATTENTION (optimisée, GPU-ready)
# ═══════════════════════════════════════════════════════════

def phase_attention(psi, n_heads, causal=True):
    L, D = psi.shape
    head_dim = D // n_heads
    heads = psi.reshape(L, n_heads, head_dim).permute(1, 0, 2)
    H, L, d = heads.shape
    
    A = heads.abs().float()
    phi = heads.angle().float()
    cos_phi = torch.cos(phi)
    sin_phi = torch.sin(phi)
    
    # cos(φ_i - φ_j) = cos(φ_i)cos(φ_j) + sin(φ_i)sin(φ_j)
    phase_scores = (cos_phi @ cos_phi.transpose(1,2) + sin_phi @ sin_phi.transpose(1,2)) / d
    A_norm_sq = (A**2).sum(dim=-1)
    amp_scores = torch.sqrt(torch.clamp(A_norm_sq.unsqueeze(2)*A_norm_sq.unsqueeze(1), min=1e-10))
    scores = phase_scores * amp_scores
    
    if causal:
        mask = torch.triu(torch.ones(L, L, device=psi.device, dtype=torch.float32), diagonal=1)
        scores = scores - 1e9 * mask.unsqueeze(0)
    
    scores = scores - scores.max(dim=-1, keepdim=True).values
    attn = torch.softmax(scores.double(), dim=-1).float()
    out = attn.to(heads.dtype) @ heads
    return out.permute(1, 0, 2).reshape(L, D).to(psi.dtype)

In [ ]:
# ═══════════════════════════════════════════════════════════
# MLP & LayerNorm
# ═══════════════════════════════════════════════════════════

def mlp(psi, W1, W2, b1=None, b2=None):
    A = psi.abs().float()
    phase = psi.angle().float()
    h = A @ W1
    if b1 is not None: h = h + b1
    h = F.relu(h)
    A_new = h @ W2
    if b2 is not None: A_new = A_new + b2
    return (A_new * (torch.cos(phase) + 1j*torch.sin(phase))).to(psi.dtype)

def layernorm_amp(psi, gamma=None, beta=None, eps=1e-6):
    A = psi.abs()
    mu = A.mean(dim=-1, keepdim=True)
    sigma = A.std(dim=-1, keepdim=True) + eps
    A_norm = (A - mu) / sigma
    if gamma is not None: A_norm = A_norm * gamma
    if beta is not None: A_norm = A_norm + beta
    phase = psi.angle()
    return (A_norm * (torch.cos(phase) + 1j*torch.sin(phase))).to(psi.dtype)

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODÈLE HWAT
# ═══════════════════════════════════════════════════════════

class HWAT(nn.Module):
    def __init__(self, vocab_size, dim=256, n_layers=4, n_heads=4, max_seq_len=64, hidden_mult=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.hidden_dim = dim * hidden_mult
        self.ctype = torch.complex64

        # Embedding déterministe (buffers = pas de gradient)
        sigma = 1.0 / math.sqrt(dim)
        def det_norm(size, seed):
            g = torch.Generator(); g.manual_seed(seed & 0xFFFFFFFF)
            return torch.randn(size, generator=g, dtype=torch.float32)
        
        A_tab = torch.zeros(vocab_size, dim)
        phi_tok = torch.zeros(vocab_size, dim)
        for tok in range(vocab_size):
            v = det_norm(dim, _fnv1a(f'amp_{tok}')) * sigma
            A_tab[tok] = v / (v.norm() + 1e-30)
            phi_tok[tok] = det_norm(dim, _fnv1a(f'phi_{tok}')).fmod(1.0) * TAU
        
        phi_pos = torch.zeros(max_seq_len, dim)
        ks = torch.arange(dim, dtype=torch.float32) / max(dim-1, 1)
        omegas = 0.1 * torch.pow(torch.tensor(math.pi/0.1), ks)
        for p in range(max_seq_len): phi_pos[p] = omegas * p
        
        self.register_buffer('A_table', A_tab)
        self.register_buffer('phi_token', phi_tok)
        self.register_buffer('phi_pos', phi_pos)

        # Blocs MLP (paramètres entraînables)
        self.W1 = nn.ParameterList()
        self.b1 = nn.ParameterList()
        self.W2 = nn.ParameterList()
        self.b2 = nn.ParameterList()
        self.ln_gamma = nn.ParameterList()
        self.ln_beta = nn.ParameterList()
        
        for lid in range(n_layers):
            H = self.hidden_dim
            D = dim
            g1 = torch.Generator(); g1.manual_seed(_fnv1a(f'mlp_w1_{lid}') & 0xFFFFFFFF)
            g3 = torch.Generator(); g3.manual_seed(_fnv1a(f'mlp_w2_{lid}') & 0xFFFFFFFF)
            lim1 = math.sqrt(3.0/D)
            lim2 = math.sqrt(3.0/H)
            self.W1.append(nn.Parameter(torch.randn(D, H, generator=g1) * 2*lim1 - lim1))
            self.b1.append(nn.Parameter(torch.zeros(H)))
            self.W2.append(nn.Parameter(torch.randn(H, D, generator=g3) * 2*lim2 - lim2))
            self.b2.append(nn.Parameter(torch.zeros(D)))
            self.ln_gamma.append(nn.Parameter(torch.ones(D)))
            self.ln_beta.append(nn.Parameter(torch.zeros(D)))

        # Tête de langage
        g = torch.Generator(); g.manual_seed(_fnv1a('lm_head') & 0xFFFFFFFF)
        self.lm_head = nn.Parameter(torch.randn(2*dim, vocab_size, generator=g) * math.sqrt(2.0/(2*dim)))
        self.lm_bias = nn.Parameter(torch.zeros(vocab_size))

    def embed(self, token_ids):
        L = min(len(token_ids), self.max_seq_len)
        token_ids = token_ids[:L]
        A = self.A_table[token_ids]
        phi_t = self.phi_token[token_ids]
        phi_p = self.phi_pos[:L]
        phi = phi_t + phi_p
        return (A * (torch.cos(phi) + 1j*torch.sin(phi))).to(self.ctype)

    def forward(self, token_ids):
        psi = self.embed(token_ids)
        for li in range(self.n_layers):
            x = layernorm_amp(psi, self.ln_gamma[li], self.ln_beta[li])
            x = phase_attention(x, self.n_heads, causal=True)
            psi = psi + x
            x = layernorm_amp(psi, self.ln_gamma[li], self.ln_beta[li])
            x = mlp(x, self.W1[li], self.W2[li], self.b1[li], self.b2[li])
            psi = psi + x
        psi_flat = torch.cat([psi.real.float(), psi.imag.float()], dim=-1)
        return psi_flat @ self.lm_head + self.lm_bias

print('✅ Modèle HWAT défini')

In [ ]:
# ═══════════════════════════════════════════════════════════
# DONNÉES D'ENTRAÎNEMENT
# ═══════════════════════════════════════════════════════════

print('Génération des données...')
rng = random.Random(42)

syns = [
    ('commencer','debuter'),('terminer','finir'),('rapide','vite'),
    ('lent','ralenti'),('grand','vaste'),('petit','minuscule'),
    ('beau','joli'),('intelligent','brillant'),('fort','puissant'),
    ('heureux','joyeux'),('triste','malheureux'),('faible','fragile'),
    ('begin','start'),('end','finish'),('fast','quick'),('slow','sluggish'),
    ('big','large'),('small','tiny'),('beautiful','pretty'),('smart','clever'),
]
rels = [
    ('Paris','capitale','France'),('Soleil','etoile','chaud'),
    ('eau','liquide','vie'),('Terre','planete','Soleil'),
    ('lumiere','onde','electromagnetique'),('Einstein','decouvert','relativite'),
    ('Newton','formule','lois du mouvement'),('Python','langage','programmation'),
    ('ADN','contient','information genetique'),('coeur','pompe','sang'),
    ('gravite','attire','objets'),('plantes','produisent','oxygene'),
    ('atome','compose','protons neutrons electrons'),('cerveau','traite','information'),
]
templates = [
    '{a} est un synonyme de {b}.', 'Le mot {a} signifie {b}.', '{a} = {b}.',
    '{a} et {b} sont equivalents.', 'On peut dire {a} ou {b}.',
    '{s} {r} {o}.', 'On sait que {s} {r} {o}.', 'C est un fait: {s} {r} {o}.',
    'La science dit: {s} {r} {o}.', 'Il est connu que {s} {r} {o}.',
]

texts = []
for _ in range(100000):
    if rng.random() < 0.6:
        a, b = rng.choice(syns)
        texts.append(rng.choice(templates[:5]).format(a=a, b=b))
    else:
        s, r, o = rng.choice(rels)
        texts.append(rng.choice(templates[5:]).format(s=s, r=r, o=o))
rng.shuffle(texts)

VOCAB_SIZE = 5000
MAX_LEN = 64
wc = Counter()
for t in texts: wc.update(t.lower().split())
w2i = {'<pad>':0, '<unk>':1, '<s>':2, '</s>':3}
for w, _ in wc.most_common(VOCAB_SIZE-4):
    w2i[w] = len(w2i)
i2w = {v:k for k,v in w2i.items()}
seqs = [[2] + [w2i.get(w,1) for w in t.lower().split()[:MAX_LEN-2]] + [3] for t in texts]

print(f'  {len(seqs):,} séquences, vocab={len(w2i)}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CRÉATION DU MODÈLE
# ═══════════════════════════════════════════════════════════

MODEL_DIM = 256
MODEL_LAYERS = 4
MODEL_HEADS = 4
BATCH_SIZE = 8
EPOCHS = 5
LR = 3e-4

model = HWAT(
    vocab_size=min(len(w2i), VOCAB_SIZE),
    dim=MODEL_DIM, n_layers=MODEL_LAYERS, n_heads=MODEL_HEADS,
    max_seq_len=MAX_LEN, hidden_mult=4
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Modèle: {n_params:,} paramètres sur {device}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ENTRAÎNEMENT
# ═══════════════════════════════════════════════════════════

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(seqs) // BATCH_SIZE
)

print(f'Entraînement: {EPOCHS} epochs, batch={BATCH_SIZE}')
print(f'Total batches: {len(seqs)//BATCH_SIZE * EPOCHS:,}')
print()

t_start = time.time()
step = 0
best_loss = float('inf')
loss_history = []

for epoch in range(EPOCHS):
    random.shuffle(seqs)
    epoch_loss = 0.0
    n_batches = len(seqs) // BATCH_SIZE
    
    for bi in range(n_batches):
        # ── Batch ──
        batch_seqs = seqs[bi*BATCH_SIZE:(bi+1)*BATCH_SIZE]
        inputs = torch.zeros(BATCH_SIZE, MAX_LEN, dtype=torch.long, device=device)
        targets = torch.zeros(BATCH_SIZE, MAX_LEN, dtype=torch.long, device=device)
        for i, s in enumerate(batch_seqs):
            n = min(len(s), MAX_LEN+1)
            inputs[i, :n-1] = torch.tensor(s[:n-1], device=device)
            targets[i, :n-1] = torch.tensor(s[1:n], device=device)
        
        # ── Forward + Backward ──
        optimizer.zero_grad()
        batch_loss = 0.0
        for b in range(BATCH_SIZE):
            logits = model(inputs[b])
            loss = F.cross_entropy(
                logits.unsqueeze(0).transpose(1, 2),
                targets[b].unsqueeze(0),
                ignore_index=0
            )
            batch_loss += loss
        batch_loss = batch_loss / BATCH_SIZE
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += batch_loss.item()
        step += 1
        
        # ── Logging ──
        if step % 200 == 0:
            elapsed = time.time() - t_start
            avg_loss = epoch_loss / (bi + 1)
            loss_history.append((step, avg_loss))
            print(f'  step {step:5d} | loss: {avg_loss:.4f} | '
                  f'{elapsed/60:.0f}min | {step/(elapsed+1e-10):.1f} step/s')
            
            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), 'model_best.pt')
        
        # ── Checkpoint ──
        if step % 2000 == 0:
            torch.save({
                'step': step, 'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, f'model_step{step}.pt')
    
    epoch_avg = epoch_loss / max(1, n_batches)
    elapsed = time.time() - t_start
    print(f'  ── Epoch {epoch+1} | avg loss: {epoch_avg:.4f} | {elapsed/60:.0f}min ──')

# ── Final ──
total_time = time.time() - t_start
torch.save(model.state_dict(), 'model_final.pt')
print(f'\n✅ Entraînement terminé')
print(f'   Steps: {step} | Temps: {total_time/60:.0f} min')
print(f'   Steps/sec: {step/total_time:.1f}')
print(f'   Best loss: {best_loss:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# TEST DE GÉNÉRATION
# ═══════════════════════════════════════════════════════════

model.eval()
print('Test de génération:')
test_prompts = ['le mot rapide', 'Paris est', 'la lumiere', 'le Soleil est']
for prompt in test_prompts:
    tokens = [w2i.get(w, 1) for w in prompt.lower().split()]
    input_ids = torch.tensor([2] + tokens, device=device)
    with torch.no_grad():
        logits = model(input_ids)
        next_token_logits = logits[-1]
        top5 = torch.topk(next_token_logits, 5)
        top_words = [i2w.get(idx.item(), '?') for idx in top5.indices]
        print(f'  "{prompt}" → {top_words}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# COURBE DE LOSS
# ═══════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
steps, losses = zip(*loss_history)
plt.figure(figsize=(10, 4))
plt.plot(steps, losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('HWAT 4.7M — Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=100)
plt.show()
print('✅ Courbe sauvegardée: loss_curve.png')